# Building a Trustworthy Housing-Price Baseline — worked solution

**Question.** How could an analyst estimate an eventual recorded property price from attributes available before that price is known? One row is a recorded observation under an educational cross-sectional assumption. This is a reference workflow for a pricing discussion, not an automatic appraisal decision. See `S18_Data_Notes.md` for field meanings and source limits.

Run from top to bottom with `S18_Housing_Workflow.csv` in the current working folder. The notebook uses Python, pandas, NumPy and scikit-learn. It learns no preparation rule from validation or reserved test records. The intended output is an auditable preparation path and a validation-only constant reference, not a high prediction score.

**Local setup:** use Python 3.12 with `python -m pip install -r S18_requirements.txt`, then select that environment's Python kernel in your Jupyter frontend. Start Jupyter with this folder as the working folder. If using a different working folder, edit `DATA_PATH` in the load cell. In a hosted notebook, upload the CSV and point `DATA_PATH` to it; that environment has not been tested here.

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| NumPy", np.__version__, "| scikit-learn", sklearn.__version__)

Python 3.12.10 | pandas 2.3.3 | NumPy 2.1.3 | scikit-learn 1.7.2


## 1. Frame the prediction and declare the inputs

`price` is the observed target (`y`), in unspecified price units. The twelve original property attributes below form the raw feature matrix (`X`). `record_id` is for audit and partition checks. `price_per_area_at_sale` uses the unknown outcome and is unavailable at the prediction moment. The legitimate `area_per_bedroom` ratio will be built later from available raw inputs inside the repeated preparation path; it is not an incoming CSV column. Mean absolute error (MAE) will summarize validation error in price units, but it cannot alone establish business suitability.

In [2]:
NUMERIC = ["area", "bedrooms", "bathrooms", "stories", "parking"]
CATEGORICAL = ["mainroad", "guestroom", "basement", "hotwaterheating",
               "airconditioning", "prefarea", "furnishingstatus"]
RAW_FEATURES = NUMERIC + CATEGORICAL
NUMERIC_WITH_RATIO = NUMERIC + ["area_per_bedroom"]
EXPECTED_COLUMNS = [
    "price", "area", "bedrooms", "bathrooms", "stories", "mainroad",
    "guestroom", "basement", "hotwaterheating", "airconditioning",
    "parking", "prefarea", "furnishingstatus", "record_id",
    "price_per_area_at_sale",
]
FORBIDDEN_INPUTS = {"price", "record_id", "price_per_area_at_sale"}
assert len(RAW_FEATURES) == 12 and not FORBIDDEN_INPUTS.intersection(RAW_FEATURES)

## 2. Load and audit the teaching data

Preserve raw tokens during import. The adapted CSV contains deliberately introduced blanks, spelling variants, parseable numeric tokens, impossible areas and eight identical copies. An ID labels an original source row and its teaching copy; it does not prove that different IDs represent different real-world properties. Verify copies before removing them. A conflicting copy would require investigation.

In [3]:
# Keep this notebook and S18_Housing_Workflow.csv in the same working folder.
# If a Jupyter service starts elsewhere, edit only this path.
DATA_PATH = Path("S18_Housing_Workflow.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH.resolve()}. Put the approved teaching CSV beside "
        "this notebook or set DATA_PATH to its local path."
    )

# Read every token as text so blank cells, comma-grouped area and count suffixes
# remain distinguishable until the fixed input-cleaning rule runs.
raw = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
assert raw.columns.tolist() == EXPECTED_COLUMNS, "Load checkpoint: unexpected columns or order"
assert raw.shape == (553, 15), "Load checkpoint: expected 553 rows and 15 columns"
assert raw["price"].ne("").all(), "Load checkpoint: a missing target cannot be an observed label"
print("Loaded rows/columns:", raw.shape)
print("Raw feature types:", raw[RAW_FEATURES].dtypes.value_counts().to_dict())

Loaded rows/columns: (553, 15)
Raw feature types: {dtype('O'): 12}


In [4]:
id_counts = raw["record_id"].value_counts()
assert raw["record_id"].ne("").all(), "Identity checkpoint: blank audit ID"
assert id_counts.max() == 2, "Identity checkpoint: an ID appears more than twice"
repeated_ids = id_counts[id_counts > 1].index
assert len(repeated_ids) == 8, "Identity checkpoint: expected eight repeated IDs"
for record_id in repeated_ids:
    group = raw.loc[raw["record_id"] == record_id]
    assert len(group.drop_duplicates()) == 1, (
        f"Identity checkpoint: {record_id} has conflicting copies; investigate before splitting"
    )
assert raw.duplicated().sum() == 8, "Identity checkpoint: copies must match across all columns"

dedup = raw.drop_duplicates(subset="record_id", keep="first").copy()
assert len(dedup) == dedup["record_id"].nunique() == 545, (
    "Identity checkpoint: one row per original audit ID is required"
)
X = dedup[RAW_FEATURES].copy()
y = pd.to_numeric(dedup["price"], errors="raise")
ids = dedup["record_id"].copy()
assert X.index.equals(y.index) and X.index.equals(ids.index), "Identity checkpoint: X/y/IDs shifted"
print("Repeated copies removed:", len(raw) - len(dedup))
print("Eligible X shape:", X.shape, "| y shape:", y.shape)

Repeated copies removed: 8
Eligible X shape: (545, 12) | y shape: (545,)


**Recovery checkpoint:** If loading or identity checks fail, confirm the working folder and exact CSV filename, then inspect the reported schema or conflicting ID. Do not silently choose one of two different rows with the same ID.

## 3. Reserve evaluation before learning preparation

Split the 545 retained records while keeping inputs, observed target and audit IDs aligned. The test portion is reserved for a later final evaluation and is not scored, browsed for target patterns or used to revise this notebook. Validation is for the declared mean/median development comparison. Training supplies fitted preparation statistics and baseline constants. Shuffling counters the source's descending-price row order, but does not prove that unseen geography, time or property identity will generalize.

In [5]:
X_dev, X_test, y_dev, y_test, id_dev, id_test = train_test_split(
    X, y, ids, test_size=0.20, shuffle=True, random_state=42
)
X_train, X_val, y_train, y_val, id_train, id_val = train_test_split(
    X_dev, y_dev, id_dev, test_size=0.25, shuffle=True, random_state=42
)
groups = [set(part) for part in (id_train, id_val, id_test)]
assert (len(X_train), len(X_val), len(X_test)) == (327, 109, 109), (
    "Split checkpoint: expected 327/109/109 after identity handling"
)
assert not (groups[0] & groups[1] or groups[0] & groups[2] or groups[1] & groups[2]), (
    "Split checkpoint: audit IDs overlap across partitions"
)
assert len(set.union(*groups)) == 545, "Split checkpoint: records were lost"
for features, target, audit_ids in (
    (X_train, y_train, id_train), (X_val, y_val, id_val), (X_test, y_test, id_test)
):
    assert features.index.equals(target.index) and features.index.equals(audit_ids.index), (
        "Split checkpoint: feature, target and audit-ID rows are misaligned"
    )
print("Train / validation / reserved test counts:", len(X_train), len(X_val), len(X_test))
print("Audit ID overlap: none")

Train / validation / reserved test counts: 327 109 109
Audit ID overlap: none


## 4. Inspect training inputs and decide what an issue means

Inspect **training** features only for data-quality decisions. A blank is different from a present but oddly formatted value. Nonpositive area is invalid; a large positive area is a flag to examine, not an automatic deletion. Zero parking is legitimate. Known category variants may be corrected by a fixed rule, while a genuinely new furnishing term stays new.

In [6]:
# This inspection uses training inputs only. Empty strings are missing markers
# here; unusual tokens have deliberately been left visible.
print("Training blank counts:", {
    field: int(X_train[field].eq("").sum())
    for field in ["area", "bathrooms", "mainroad", "furnishingstatus"]
})
print("Training area tokens needing review:",
      sorted(set(X_train.loc[X_train["area"].str.contains(",", regex=False)
                              | X_train["area"].isin(["0", "-1", "-10"]), "area"])))
print("Training bathroom count-suffix tokens:",
      sorted(set(X_train.loc[X_train["bathrooms"].str.contains(" bath", regex=False), "bathrooms"])))
print("Training mainroad categories:", sorted(set(X_train["mainroad"])))
print("Training furnishing categories:", sorted(set(X_train["furnishingstatus"])))
print("Training zero-parking rows:", int(X_train["parking"].eq("0").sum()))

Training blank counts: {'area': 3, 'bathrooms': 3, 'mainroad': 3, 'furnishingstatus': 2}
Training area tokens needing review: ['-1', '-10', '3,300', '3,512', '5,400']
Training bathroom count-suffix tokens: ['1 bath', '2 baths']
Training mainroad categories: ['', ' YES ', 'no', 'yes']
Training furnishing categories: ['', 'Furnished', 'furnished', 'semi furnished', 'semi-furnished', 'unfurnished', 'unfurnshed']
Training zero-parking rows: 181


## 5. Make one repeatable, row-preserving input rule

This helper sees only twelve raw feature columns. It uses fixed, documented rules: valid comma thousands grouping in `area`; an integer followed by `bath`/`baths` in `bathrooms`; case and outer-space normalization for yes/no; two known furnishing spelling fixes; invalid nonpositive area/counts to missing, with parking zero retained. Unknown numeric tokens raise an error. A well-formed new furnishing term remains new for the fitted encoder. The helper does not estimate medians, modes, scales or category vocabularies.

In [7]:
def parse_whole_number(value, field):
    """Parse only declared integer forms. Blank -> missing; unknown token -> error."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        if not np.isfinite(value) or not float(value).is_integer():
            raise ValueError(f"{field}: expected a finite whole number, got {value!r}")
        return float(value)
    token = str(value).strip().lower()
    if token == "":
        return np.nan
    if field == "area" and re.fullmatch(r"\d{1,3}(?:,\d{3})+", token):
        token = token.replace(",", "")
    if field == "bathrooms" and re.fullmatch(r"\d+ baths?", token):
        token = token.split()[0]
    if not re.fullmatch(r"[+-]?\d+", token):
        raise ValueError(f"{field}: unrecognized numeric token {value!r}")
    return float(int(token))


def clean_category(value, field):
    """Apply fixed known corrections; preserve a well-formed unseen furnishing term."""
    if pd.isna(value):
        return np.nan
    token = str(value).strip().lower()
    if token == "":
        return np.nan
    if field == "furnishingstatus":
        token = {"semi furnished": "semi-furnished",
                 "unfurnshed": "unfurnished"}.get(token, token)
        if not re.fullmatch(r"[a-z]+(?:-[a-z]+)*", token):
            raise ValueError(f"{field}: unrecognized category format {value!r}")
        return token  # A new well-formed term stays new for the encoder.
    if token not in {"yes", "no"}:
        raise ValueError(f"{field}: expected yes/no or missing, got {value!r}")
    return token


def clean_features(frame):
    """Row-preserving, target-independent rules for the twelve raw inputs."""
    if not isinstance(frame, pd.DataFrame) or frame.columns.tolist() != RAW_FEATURES:
        raise ValueError("Expected a DataFrame with exactly the twelve ordered raw feature columns")
    cleaned = frame.copy()
    for field in NUMERIC:
        cleaned[field] = cleaned[field].map(
            lambda value, field=field: parse_whole_number(value, field)
        ).astype(float)
    # These are fixed domain rules, not statistics learned from other rows.
    for field in ["area", "bedrooms", "bathrooms", "stories"]:
        cleaned.loc[cleaned[field] <= 0, field] = np.nan
    cleaned.loc[cleaned["parking"] < 0, "parking"] = np.nan  # zero is valid
    for field in CATEGORICAL:
        cleaned[field] = cleaned[field].map(
            lambda value, field=field: clean_category(value, field)
        )
    cleaned["area_per_bedroom"] = cleaned["area"] / cleaned["bedrooms"]
    cleaned.loc[~np.isfinite(cleaned["area_per_bedroom"]), "area_per_bedroom"] = np.nan
    assert len(cleaned) == len(frame) and cleaned.index.equals(frame.index)
    return cleaned


def names_with_ratio(transformer, input_features):
    return np.asarray(list(input_features) + ["area_per_bedroom"], dtype=object)

In [8]:
clean_train = clean_features(X_train)
assert clean_train.columns.tolist() == RAW_FEATURES + ["area_per_bedroom"]
assert len(clean_train) == len(X_train) and clean_train.index.equals(X_train.index)
assert clean_train["area"].isna().any() and clean_train["area_per_bedroom"].isna().any()
assert (clean_train["parking"] == 0).any(), "Zero parking must stay valid"

zero_bedrooms = X_train.iloc[[0]].copy()
zero_bedrooms.loc[:, "bedrooms"] = "0"
assert pd.isna(clean_features(zero_bedrooms)["area_per_bedroom"].iloc[0])
try:
    malformed = X_train.iloc[[0]].copy()
    malformed.loc[:, "area"] = "about 4000"
    clean_features(malformed)
except ValueError as error:
    print("Unrecognized numeric input rejected:", error)
else:
    raise AssertionError("Input checkpoint: unknown numeric token was silently accepted")
print("Cleaned training rows/columns:", clean_train.shape)

Unrecognized numeric input rejected: area: unrecognized numeric token 'about 4000'
Cleaned training rows/columns: (327, 13)


The derived `area_per_bedroom` is missing when its numerator or denominator is missing or invalid. The next imputer learns how to fill it from **training** rows. The following IQR threshold is a training-only inspection aid; it does not delete data or define a universal property-size limit.

In [9]:
area_train = clean_train["area"].dropna()
q1, q3 = area_train.quantile([0.25, 0.75])
iqr = q3 - q1
upper_flag = q3 + 1.5 * iqr
flagged = clean_train["area"].gt(upper_flag)
print(f"Training-only area IQR upper flag: {upper_flag:.1f} recorded area units")
print("Flagged training rows:", int(flagged.sum()),
      "| largest valid training area:", area_train.max())
print("No row is removed just because its area is unusually large.")

Training-only area IQR upper flag: 10612.5 recorded area units
Flagged training rows: 7 | largest valid training area: 16200.0
No row is removed just because its area is unusually large.


## 6. Fit mixed-data preparation with the median reference

Raw features → fixed row rules and safe ratio → numerical median imputation and standardization, plus categorical most-frequent imputation and one-hot encoding → constant baseline. A configured transformer has not learned anything until `fit` runs. Here the full pipeline fits on training inputs and targets only. Explicit columns and `remainder="drop"` keep the outcome, ID and leakage illustration out of the estimator.

In [10]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

In [11]:
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

In [12]:
columns = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_WITH_RATIO),
    ("cat", categorical_pipe, CATEGORICAL),
], remainder="drop")
median_model = Pipeline([
    ("clean", FunctionTransformer(clean_features, validate=False,
                                  feature_names_out=names_with_ratio)),
    ("columns", columns),
    ("baseline", DummyRegressor(strategy="median")),
])
median_model.fit(X_train, y_train)
print("Median baseline pipeline fitted on training rows only.")

Median baseline pipeline fitted on training rows only.

In [13]:
prepared = median_model[:-1]
train_matrix = prepared.transform(X_train)
fitted_columns = median_model.named_steps["columns"]
num_imputer = fitted_columns.named_transformers_["num"].named_steps["impute"]
cat_pipeline = fitted_columns.named_transformers_["cat"]
cat_imputer = cat_pipeline.named_steps["impute"]
encoder = cat_pipeline.named_steps["encode"]
feature_names = prepared.get_feature_names_out()

assert train_matrix.shape == (len(X_train), len(feature_names)), (
    "Preparation checkpoint: transformed row count/feature names disagree"
)
assert np.isfinite(train_matrix).all(), "Preparation checkpoint: missing/nonfinite transformed values"
assert np.allclose(num_imputer.statistics_, clean_train[NUMERIC_WITH_RATIO].median().to_numpy())
assert len(feature_names) == len(set(feature_names))
print("Prepared training matrix:", train_matrix.shape)
print("Training-fitted numeric medians:", dict(zip(NUMERIC_WITH_RATIO, num_imputer.statistics_)))
print("Training-fitted categorical modes:", dict(zip(CATEGORICAL, cat_imputer.statistics_)))
print("Training-learned furnishing categories:", encoder.categories_[-1].tolist())
print("Transformed feature names:", feature_names.tolist())

# transform must reuse the fitted state, not learn from validation rows.
medians_before = num_imputer.statistics_.copy()
scale_mean_before = fitted_columns.named_transformers_["num"].named_steps["scale"].mean_.copy()
categories_before = [values.copy() for values in encoder.categories_]
val_matrix = prepared.transform(X_val)
assert val_matrix.shape == (len(X_val), len(feature_names)) and np.isfinite(val_matrix).all()
assert np.array_equal(num_imputer.statistics_, medians_before)
assert np.array_equal(fitted_columns.named_transformers_["num"].named_steps["scale"].mean_, scale_mean_before)
assert all(np.array_equal(now, before) for now, before in zip(encoder.categories_, categories_before))
print("Validation transformed with unchanged training-fitted settings:", val_matrix.shape)

Prepared training matrix: (327, 21)
Training-fitted numeric medians: {'area': np.float64(4500.0), 'bedrooms': np.float64(3.0), 'bathrooms': np.float64(1.0), 'stories': np.float64(2.0), 'parking': np.float64(0.0), 'area_per_bedroom': np.float64(1673.3333333333335)}
Training-fitted categorical modes: {'mainroad': 'yes', 'guestroom': 'no', 'basement': 'no', 'hotwaterheating': 'no', 'airconditioning': 'no', 'prefarea': 'no', 'furnishingstatus': 'semi-furnished'}
Training-learned furnishing categories: ['furnished', 'semi-furnished', 'unfurnished']
Transformed feature names: ['num__area', 'num__bedrooms', 'num__bathrooms', 'num__stories', 'num__parking', 'num__area_per_bedroom', 'cat__mainroad_no', 'cat__mainroad_yes', 'cat__guestroom_no', 'cat__guestroom_yes', 'cat__basement_no', 'cat__basement_yes', 'cat__hotwaterheating_no', 'cat__hotwaterheating_yes', 'cat__airconditioning_no', 'cat__airconditioning_yes', 'cat__prefarea_no', 'cat__prefarea_yes', 'cat__furnishingstatus_furnished', 'cat__

Validation transformed with unchanged training-fitted settings: (109, 21)


**Recovery checkpoint:** If the transformed width, names, finite-value check or medians disagree, inspect the fixed cleaning output and the fitted branch steps. A constant baseline's score cannot diagnose an incorrectly encoded feature: it predicts the same value regardless of the property attributes.

## 7. Evaluate the median reference on validation only

The median baseline predicts the training-target median for every validation row. MAE is the mean size of absolute errors, measured in the original price units; it is not an accuracy percentage.

In [14]:
median_predictions = median_model.predict(X_val)
median_mae = mean_absolute_error(y_val, median_predictions)
median_constant = float(np.median(y_train))
assert np.allclose(median_predictions, median_constant), (
    "Baseline checkpoint: median predictions differ from the training median"
)
assert np.isclose(median_mae, np.mean(np.abs(y_val.to_numpy() - median_predictions))), (
    "Baseline checkpoint: MAE does not equal mean absolute error"
)
print(f"Training median constant: {median_constant:,.2f} price units")
print(f"Median validation MAE: {median_mae:,.2f} price units")

Training median constant: 4,305,000.00 price units
Median validation MAE: 1,301,903.67 price units


Now send one **synthetic**, feature-only input through the same fitted object. Its area is missing and its furnishing term is well formed but was not observed in training. The encoder's furnishing block should be all zeros for that unseen term; this fallback does not mean the model understands the new term.

In [15]:
# Synthetic new request, separate from all recorded partitions. The input has
# exactly the twelve raw features: no price, audit ID, leakage field or ratio.
fresh = pd.DataFrame([{
    "area": "", "bedrooms": "3", "bathrooms": "2", "stories": "2",
    "parking": "0", "mainroad": "yes", "guestroom": "no",
    "basement": "no", "hotwaterheating": "no", "airconditioning": "yes",
    "prefarea": "no", "furnishingstatus": "part-furnished",
}], columns=RAW_FEATURES)
assert fresh.columns.tolist() == RAW_FEATURES and not FORBIDDEN_INPUTS.intersection(fresh)
assert pd.isna(clean_features(fresh)["area_per_bedroom"].iloc[0])
assert "part-furnished" not in encoder.categories_[-1]
fresh_clean = clean_features(fresh)
encoded_categories = cat_pipeline.transform(fresh_clean[CATEGORICAL])
start = sum(len(values) for values in encoder.categories_[:-1])
furnishing_block = encoded_categories[0, start:]
assert furnishing_block.shape[0] == len(encoder.categories_[-1])
assert np.all(furnishing_block == 0), "New-category checkpoint: expected all-zero furnishing block"
fresh_prediction = median_model.predict(fresh)
assert np.isfinite(fresh_prediction).all()
print("Synthetic fresh-row prediction:", fresh_prediction[0], "price units")
print("Unseen furnishing category's encoded block:", furnishing_block.tolist())

Synthetic fresh-row prediction:

 4305000.0 price units
Unseen furnishing category's encoded block: [0.0, 0.0, 0.0]


## 8. Compare an independent mean reference

A clone gives the mean candidate its own unfitted preprocessing objects. It uses the **same** training and validation records. Select the lower validation MAE; choose median on an exact tie. A median is optimal among constants on its own training targets for absolute loss, but need not win on this validation partition.

In [16]:
# clone creates new unfitted preprocessing and estimator objects; it does not
# reuse the fitted median pipeline's statistics.
mean_model = clone(median_model).set_params(baseline__strategy="mean")
assert mean_model.named_steps["columns"] is not median_model.named_steps["columns"]
mean_model.fit(X_train, y_train)
mean_predictions = mean_model.predict(X_val)
mean_mae = mean_absolute_error(y_val, mean_predictions)
mean_constant = float(np.mean(y_train))
assert np.allclose(mean_predictions, mean_constant)
assert np.isclose(mean_mae, np.mean(np.abs(y_val.to_numpy() - mean_predictions)))
preferred = "median" if median_mae <= mean_mae else "mean"  # median wins an exact tie
print(f"Training mean constant: {mean_constant:,.2f} price units")
print(f"Mean validation MAE: {mean_mae:,.2f} price units")
print("Development reference selected by lower validation MAE (median on tie):", preferred)

Training mean constant: 4,713,074.74 price units


Mean validation MAE: 1,345,292.84 price units
Development reference selected by lower validation MAE (median on tie): median


**What the result supports.** The identity checks, split, fixed cleaning rules, fitted transformation checks, fresh-input behavior and fair validation comparison ran as declared. Neither constant baseline learns property-specific relationships. Changing encoding or scaling cannot improve a fixed constant prediction, so these scores do not prove the transformed features are useful. They also do not establish causal effects, a future test score, a representative housing market or deployment readiness.

**Cross-validation bridge (conceptual only):** keep the reserved test band aside. Within development data, rotate which fold plays validation: `A+B → C`, `A+C → B`, `B+C → A`. Refit the **entire** cleaning/preprocessing/estimator pipeline on each fold's training portion. Do not preprocess all development rows once and then rotate folds. Executed cross-validation and a feature-learning regression estimator belong to the upcoming regression work.

**Technical references checked 18 November 2026:** [data leakage and pipelines](https://scikit-learn.org/stable/common_pitfalls.html), [splitting](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), [composite estimators](https://scikit-learn.org/stable/modules/compose.html), [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), [FunctionTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html), [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html), [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html), [DummyRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html), [MAE](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html). The code cell above reports the versions actually used when this notebook runs.

Additional environment/CSV references checked 18 November 2026: [Python virtual environments](https://docs.python.org/3.12/library/venv.html), [Jupyter installation](https://jupyter.org/install), [IPython kernel setup](https://ipython.readthedocs.io/en/stable/install/kernel_install.html), [pandas CSV parsing](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).